## 🥇 Gold Layer – Business KPIs & Advanced Analytics

### 🎯 Objective

The Gold layer provides curated, business-ready datasets by combining **real-time streaming data** with **historical batch data** to enable comprehensive analytics and decision-making.

---

### 🏗️ Data Sources Integrated

* **Streaming Data (Kafka → Bronze → Silver)**
  Real-time order events

* **Batch Data (CSV Upload → Table)**
  Historical sales data (`smartgear_sales`)

👉 Both datasets are **schema-aligned and unified** to create a complete analytical view.

---

### 📊 KPIs Implemented

* Region-wise total revenue
* Month-over-Month (MoM) growth *(based on combined data)*
* Average order value per region
* Store performance segmentation

---

### 🔍 Advanced Analytics

* Top 5 products by revenue
* Rolling revenue (7-day moving average using historical + streaming data)
* Anomaly detection (±30% deviation from average revenue)

---

### 📌 Outcome

This layer transforms cleaned data into:

* Executive dashboards
* Strategic insights
* Decision-making metrics

---

### 💡 Key Design Insight

By integrating **batch (historical)** and **streaming (real-time)** data, the system enables:

* Trend analysis over time
* Real-time monitoring
* Data validation and reconciliation

This hybrid approach reflects real-world data engineering practices.


In [0]:
from pyspark.sql.functions import avg, to_date, rank, sum as _sum, col, lower, trim, to_timestamp
from pyspark.sql.window import Window

In [0]:
#Read data from Silver Layer
df_stream = spark.read.table("workspace.smartgear.silver_orders") \
    .select("timestamp", "product", "region", "revenue")
df_batch = spark.read.table("workspace.smartgear.smartgear_sales")

df_batch_clean = df_batch \
    .withColumn("timestamp", to_timestamp("OrderDate")) \
    .withColumn("region", lower(trim(col("Region")))) \
    .withColumn("product", col("Product")) \
    .withColumn("revenue", col("Quantity") * col("UnitPrice")) \
    .select("timestamp", "product", "region", "revenue")

df = df_stream.unionByName(df_batch_clean)


In [0]:
#KPI 1: Region-wise Total Revenue
df_region_revenue = df.groupBy("region") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "total_revenue")

#KPI 2: Average Order Value per Region
df_avg_order = df.groupBy("region") \
    .agg(avg("revenue").alias("avg_order_value"))

#KPI 3: Daily Revenue Trend
df_daily = df.withColumn("date", to_date("timestamp")) \
    .groupBy("date") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "daily_revenue") \
    .orderBy("date")

#KPI 4: Top 3 Stores per Region
window_spec = Window.partitionBy("region").orderBy(col("revenue").desc())

df_top_stores = df.withColumn("rank", rank().over(window_spec)) \
    .filter("rank <= 3")

#KPI 5: Region Contribution %
total_revenue = df.agg(_sum("revenue")).collect()[0][0]

df_region_pct = df_region_revenue.withColumn(
    "percentage",
    (col("total_revenue") / total_revenue) * 100
)

In [0]:

#Top 5 Products by Revenue
window_product = Window.orderBy(col("total_revenue").desc())

df_products = df.groupBy("product") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "total_revenue") \
    .withColumn("rank", rank().over(window_product)) \
    .filter("rank <= 5")

#Rolling Revenue (7-Day Avg)
window_7day = Window.orderBy("date").rowsBetween(-6, 0)

df_rolling = df_daily.withColumn(
    "rolling_avg",
    avg("daily_revenue").over(window_7day)
)

#Anomaly Detection (±30%)
avg_rev = df_daily.agg(avg("daily_revenue")).collect()[0][0]

df_anomaly = df_daily.withColumn(
    "anomaly",
    (col("daily_revenue") > avg_rev * 1.3) |
    (col("daily_revenue") < avg_rev * 0.7)
)



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
#Saving to Gold Tables
df_region_revenue.write.mode("overwrite").saveAsTable("workspace.smartgear.gold_region_revenue")
df_avg_order.write.mode("overwrite").saveAsTable("workspace.smartgear.gold_avg_order")
df_daily.write.mode("overwrite").saveAsTable("workspace.smartgear.gold_daily_revenue")
df_top_stores.write.mode("overwrite").saveAsTable("workspace.smartgear.gold_top_stores")
df_region_pct.write.mode("overwrite").saveAsTable("workspace.smartgear.gold_region_pct")
df_products.write.mode("overwrite").saveAsTable("workspace.smartgear.gold_top_products")
df_rolling.write.mode("overwrite").saveAsTable("workspace.smartgear.gold_rolling_revenue")
df_anomaly.write.mode("overwrite").saveAsTable("workspace.smartgear.gold_anomaly_detection")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
